In [ ]:
%%capture
%pip install pandas
%pip install torch
%pip install sentence-transformers
%pip install numpy

In [ ]:
import pandas as pd
import torch
import numpy as np
from typing import Literal
from sentence_transformers import SentenceTransformer

## Build the Graph Neural Network

First let's get our cleaned dataset so that we can initialize the nodes and edges of our graph.

In [ ]:
dataset = pd.read_csv("data/clean.csv")

In [ ]:
dataset

### Build Interaction Edges

Each edge represents a directed message from one user to another. There are two ways an interaction is determined:
- Temporal interaction: pair of messages within a set time window between two **different** users
    - Lots of noise, but on average these are unique, true interactions
- Mention interaction: explicit mention to another user

**There may be a way to weight these interaction types becauase mentions are explicit and hence more confident as meaningful edges.**

First we need a way to convert discord usernames to IDs.

In [ ]:
user_df = dataset[["AuthorID", "Author"]].drop_duplicates().reset_index(drop=True)
username_to_id = user_df.set_index("Author")["AuthorID"].to_dict()
username_to_id

- Drop duplicate users (Authors) so we only have the unique users
- Reset the indices from `0-N` (`N` = number of users) and drop the old index column

Now we can find edges in our cleaned dataset.

Each edge will have the following information:
- edge_id
- source_user_id
- target_user_id
- timestamp
- edge_type (mention | temporal)
- metadata (time_delta, has_media)

In [ ]:
edges = []
T = pd.Timedelta(minutes=5)
last_index = None

In [ ]:
for i, row in dataset.iterrows():
    source_user = row["AuthorID"]
    timestamp = row["Date"]
    content = row["Content"]

    # mention edges
    if row["has_text"]:
        for username, target in username_to_id.items():
            if f"@{username}" in content and target != source_user:
                edges.append({
                    "source_user_id": source_user,
                    "target_user_id": target,
                    "timestamp": timestamp,
                    "edge_type": "mention",
                    #"metadata": {"has_media": bool} add later in v2 maybe
                })
    # temporal edges
    if last_index is not None:
        prev = dataset.loc[last_index]
        prev_source = prev["AuthorID"]
        prev_timestamp = prev["Date"]

        if source_user != prev_source and (timestamp - prev_timestamp) <= T:
            edges.append({
                "source_user_id": source_user,
                "target_user_id": prev_source,
                "timestamp": timestamp,
                "edge_type": "temporal",
                #"metadata": {"has_media": bool} add later in v2 maybe
            })
    
    last_index = i

In [ ]:
edges_df = pd.DataFrame(edges)
edges_df

### Build User Nodes

Each node will represent a user and we'll use `pandas` Dataframes to represent them.

First let's get the `avg_text_embedding` of each user.

In [ ]:
text_df = dataset[dataset["has_text"]].copy()
text_df

Clean the text content for links, this is **only** for embedding. We may need links in text for downstream tasks.

In [ ]:
URL_PATTERN = r"https?://\S+"

text_df["clean_text"] = (
    text_df["Content"]
    .str.replace(URL_PATTERN, "", regex=True)
    .str.strip()
)

Now we can use the `clean_text` column to create average text embeddings for each user node.

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(
    text_df["clean_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
)

text_df["embeddings"] = list(embeddings)

In [ ]:
text_df.head()

Now we have an `embeddings` column that embeds the cleaned content from each message, let's take the average.

In [ ]:
avg_text_embedding = (
    text_df
    .groupby("AuthorID")["embeddings"]
    .apply(lambda x: np.mean(np.stack(x), axis=0))
)

This cell takes the `text_df`, groups it by `AuthorID` (user IDs) as indices that point to a `Series` of embeddings.

```
AuthorID → [emb₁, emb₂, emb₃, ...]
```

Then we use the `apply` method with a lambda function with parameter `x`, a `Series` of embeddings.

```
x = [array(d), array(d), array(d), ...]
x: Series[array(d), array(d), ...]
```

The lambda function stacks each array (embedding) from a user and gets the mean embedding by taking the mean of `axis=0`.
We use `stack` because NumPy can't get the mean array of a Series or arrays, hence we need to make a 2D array.

For example:
```python
x = Series[
  [0.1, 0.2, 0.3],
  [0.0, 0.4, 0.1],
  [0.2, 0.1, 0.5]
]

np.stack(x)

x = array([
  [0.1, 0.2, 0.3],
  [0.0, 0.4, 0.1],
  [0.2, 0.1, 0.5]
])

np.mean(np.stack(x), axis=0) # store the means of the first dimension (axis=0)

y = array[
    avg([0.1, 0.0, 0.2]),
    avg([0.2, 0.4, 0.1]),
    avg([0.3, 0.1, 0.5])
]
```

This gives us `y`, the average embedding for that user node.

In [ ]:
avg_text_embedding = (
    text_df
    .groupby("AuthorID")["embeddings"]
    .apply(lambda x: np.mean(np.stack(x), axis=0))
)
avg_text_embedding

Now we can create our `pandas` DataFrame that we'll use to store our user nodes.

In [ ]:
nodes_df = user_df.merge(
    avg_text_embedding,
    left_on="AuthorID", # join by matching AuthorID to...
    right_index=True, # match to the index of avg_text_embedding
    how="left" # keep all users in user_df, give NaN if no matching user in avg_textembedding
).rename(
    columns={
        "AuthorID": "user_id",
        "Author": "username",
        "embeddings": "avg_text_embedding",
    }
)
nodes_df

Now let's get the following activity stats from `edges_df`:
- interaction initiation count
- interaction reply count
- total message count

First we get the counts of outgoing and incoming edges for each user.

In [ ]:
source_df = edges_df[["source_user_id"]]
source_df.value_counts()

In [ ]:
target_df = edges_df[["target_user_id"]]
target_df.value_counts()

Now we get a Series of counts by getting the `value_counts` of the indexed column in each `source` and `target` DataFrame.

In [ ]:
source_counts = source_df["source_user_id"].value_counts()
target_counts = target_df["target_user_id"].value_counts()

The counts are in Series so they can be mapped to the new feature columns `initiation_count` and `target_count` in the `nodes_df`.

In [ ]:
nodes_df["initiation_count"] = (
    nodes_df["user_id"]
    .map(source_counts)
    .fillna(0)
    .astype(int)
)

nodes_df["target_count"] = (
    nodes_df["user_id"]
    .map(target_counts)
    .fillna(0)
    .astype(int)
)

See below for how the `map` method works.
- It uses the `user_id` from `nodes_df` as the index to the counts Series
- The new column in `nodes_df`is set to this mapping because it's in the same order as the `user_id` column in the Dataframe.

In [ ]:
nodes_df["user_id"].map(source_counts)

In [ ]:
nodes_df["user_id"].map(target_counts)

Finally we add the two interaction types together to get the `tota_interactions` for each user node. Save it to its own column in `nodes_df`.

In [ ]:
nodes_df["total_interactions"] = nodes_df["initiation_count"] + nodes_df["target_count"]
nodes_df